# Social Media Intelligence with XPOZ MCP

This cookbook shows how to use [XPOZ](https://xpoz.ai) — a social-intelligence MCP server with 1.5B+ indexed posts across Twitter, Instagram, Reddit & TikTok — to build a real-time social media analysis pipeline with Claude.

**What you'll learn:**
- Connect to a remote MCP server via HTTP (JSON-RPC)
- Fetch cross-platform social data (Twitter, Reddit, Instagram)
- Use Claude to produce structured sentiment analysis from raw social posts

**Prerequisites:**
- XPOZ API key (free at [xpoz.ai](https://xpoz.ai) — 100K results/month)
- Anthropic API key ([console.anthropic.com](https://console.anthropic.com))

## Setup

Install dependencies and configure API keys. XPOZ exposes an MCP server at `https://mcp.xpoz.ai/mcp` — we'll call it via direct HTTP using the JSON-RPC protocol, which avoids MCP SDK streaming issues in notebook environments.

In [ ]:
%%capture
%pip install -q anthropic httpx python-dotenv

In [ ]:
import anthropic
import json
import re
import httpx
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
XPOZ_API_KEY = os.environ["XPOZ_API_KEY"]

MODEL = "claude-sonnet-4-6"

## Connect to XPOZ MCP Server

XPOZ provides social data via the [Model Context Protocol](https://modelcontextprotocol.io/). The MCP server accepts JSON-RPC requests over HTTP. We create a lightweight client that:

1. Initializes a session (handshake)
2. Calls tools like `countTweets`, `getTwitterPostsByKeywords`, etc.
3. Parses the response (XPOZ returns a compact tabular format)

In [ ]:
XPOZ_URL = "https://mcp.xpoz.ai/mcp"
_msg_id = 0
_session_id = None


def _parse_sse(text):
    """Extract JSON-RPC result from an SSE response."""
    result = {}
    for line in text.split("\n"):
        if line.startswith("data: "):
            try:
                result = json.loads(line[6:])
            except json.JSONDecodeError:
                pass
    return result


async def _mcp_post(client, payload):
    """POST a JSON-RPC request and handle JSON or SSE responses."""
    global _session_id
    headers = {
        "Authorization": f"Bearer {XPOZ_API_KEY}",
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
    }
    if _session_id:
        headers["Mcp-Session-Id"] = _session_id
    resp = await client.post(XPOZ_URL, json=payload, headers=headers)
    if "mcp-session-id" in resp.headers:
        _session_id = resp.headers["mcp-session-id"]
    ct = resp.headers.get("content-type", "")
    if "text/event-stream" in ct:
        return _parse_sse(resp.text)
    elif resp.text.strip():
        return resp.json()
    return {}

In [ ]:
async def _ensure_session(client):
    """Initialize the MCP session if not already active."""
    global _session_id
    if _session_id:
        return
    await _mcp_post(client, {
        "jsonrpc": "2.0",
        "id": 0,
        "method": "initialize",
        "params": {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "cookbook", "version": "1.0"},
        },
    })
    await _mcp_post(client, {
        "jsonrpc": "2.0",
        "method": "notifications/initialized",
    })

In [ ]:
import csv
import io


def _parse_xpoz(text):
    """Parse XPOZ compact tabular format into a Python dict.

    XPOZ returns results in a space-efficient CSV-like format:
        results[N]{field1,field2,...}:
            val1,val2,...
    """
    result = {"success": "success: true" in text, "data": {}}
    hdr = re.search(r"results\[\d+\]\{([^}]+)\}:", text)
    if hdr:
        fields = hdr.group(1).split(",")
        rows = []
        for line in text[hdr.end() :].split("\n"):
            if not line.startswith("    "):
                if rows or (line.strip() and not line.startswith(" ")):
                    break
                continue
            try:
                vals = next(csv.reader(io.StringIO(line.strip())))
                row = {}
                for i, f in enumerate(fields):
                    if i < len(vals):
                        v = vals[i].strip()
                        if v in ("null", ""):
                            row[f] = None
                        else:
                            try:
                                row[f] = int(v)
                            except ValueError:
                                try:
                                    row[f] = float(v)
                                except ValueError:
                                    row[f] = v
                rows.append(row)
            except Exception:
                pass
        result["data"]["results"] = rows
    for m in re.finditer(r"^\s{2}(\w+):\s+(.+)$", text, re.MULTILINE):
        k, v = m.group(1), m.group(2).strip()
        if k.startswith("results"):
            continue
        try:
            result["data"][k] = int(v)
        except ValueError:
            try:
                result["data"][k] = float(v)
            except ValueError:
                result["data"][k] = v
    return result

In [ ]:
async def call_xpoz(tool_name, params):
    """Call an XPOZ MCP tool and return parsed results."""
    global _msg_id
    _msg_id += 1
    async with httpx.AsyncClient(timeout=120) as client:
        await _ensure_session(client)
        data = await _mcp_post(client, {
            "jsonrpc": "2.0",
            "id": _msg_id,
            "method": "tools/call",
            "params": {"name": tool_name, "arguments": params},
        })
    if "error" in data:
        print(f"MCP error: {data['error']}")
        return {"data": {"results": []}}
    if "result" not in data:
        print(f"Unexpected response: {str(data)[:200]}")
        return {"data": {"results": []}}
    content = data.get("result", {}).get("content", [{}])
    text = content[0].get("text", "{}") if content else "{}"
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return _parse_xpoz(text)

Verify the connection by running a quick count query.

In [ ]:
def days_ago(n):
    return (datetime.now() - timedelta(days=n)).strftime("%Y-%m-%d")


test = await call_xpoz("countTweets", {"phrase": "Bitcoin", "startDate": days_ago(1)})
print(f"Connected to XPOZ MCP — Bitcoin tweets in last 24h: {test['data'].get('count', 'ok'):,}")

## Fetch Social Data

We'll analyze **Bitcoin** social sentiment across three platforms. XPOZ provides separate tools for each:

| Tool | Platform | What it returns |
|---|---|---|
| `countTweets` | Twitter | Total volume for a keyword |
| `getTwitterPostsByKeywords` | Twitter | Post text, author, engagement |
| `getRedditPostsByKeywords` | Reddit | Title, selftext, score, subreddit |
| `getInstagramPostsByKeywords` | Instagram | Caption, username, likes |

In [ ]:
# Total tweet volume (7 days)
count_r = await call_xpoz("countTweets", {
    "phrase": "Bitcoin",
    "startDate": days_ago(7),
})
tweet_count = count_r["data"].get("count", 0)
print(f"7-day Bitcoin tweet volume: {tweet_count:,}")

In [ ]:
# Twitter — two batches to get ~600 posts (API caps at 300 per call)
tw1 = await call_xpoz("getTwitterPostsByKeywords", {
    "query": "Bitcoin OR BTC",
    "startDate": days_ago(3),
    "limit": 300,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"],
})
tw2 = await call_xpoz("getTwitterPostsByKeywords", {
    "query": "Bitcoin OR BTC",
    "startDate": days_ago(7),
    "endDate": days_ago(3),
    "limit": 300,
    "fields": ["text", "authorUsername", "likeCount", "retweetCount", "createdAtDate"],
})
tweets = tw1["data"].get("results", []) + tw2["data"].get("results", [])
print(f"Twitter: {len(tweets)} posts")

In [ ]:
# Reddit
rd_r = await call_xpoz("getRedditPostsByKeywords", {
    "query": "Bitcoin OR BTC",
    "startDate": days_ago(7),
    "limit": 300,
    "fields": ["title", "selftext", "authorUsername", "score", "subredditName", "createdAtDate"],
})
reddit_posts = rd_r["data"].get("results", [])
print(f"Reddit: {len(reddit_posts)} posts")

In [ ]:
# Instagram (no date filter — sparser data)
ig_r = await call_xpoz("getInstagramPostsByKeywords", {
    "query": "Bitcoin",
    "limit": 100,
    "fields": ["caption", "username", "likeCount", "commentCount", "createdAtDate"],
})
ig_posts = ig_r["data"].get("results", [])
print(f"Instagram: {len(ig_posts)} posts")

In [ ]:
total_posts = len(tweets) + len(reddit_posts) + len(ig_posts)
print(f"\nTotal: {total_posts} posts sampled from {tweet_count:,} indexed tweets")

## Analyse with Claude

Feed the social data to Claude and ask for a structured sentiment analysis. The prompt requests a specific output format so results are consistent and parseable.

In [ ]:
def _safe_int(v):
    """Safely convert a value to int for display."""
    try:
        return int(v) if v else 0
    except (ValueError, TypeError):
        return 0


tw_text = "\n".join(
    f"@{p.get('authorUsername', '?')}: {p.get('text', '')} "
    f"[likes:{_safe_int(p.get('likeCount'))}, RTs:{_safe_int(p.get('retweetCount'))}]"
    for p in tweets[:200]
)
rd_text = "\n".join(
    f"r/{p.get('subredditName', '')} — {p.get('title', '')}: "
    f"{(p.get('selftext') or '')[:200]} [score:{_safe_int(p.get('score'))}]"
    for p in reddit_posts[:100]
)
ig_text = "\n".join(
    f"@{p.get('username', '?')}: {(p.get('caption') or '')[:200]} "
    f"[likes:{_safe_int(p.get('likeCount'))}]"
    for p in ig_posts[:50]
)

In [ ]:
prompt = f"""Analyse the social-media sentiment for Bitcoin based on these posts from the last 7 days.
Total tweet volume: {tweet_count:,}

=== Twitter ({len(tweets)} posts) ===
{tw_text}

=== Reddit ({len(reddit_posts)} posts) ===
{rd_text}

=== Instagram ({len(ig_posts)} posts) ===
{ig_text}

Return your analysis in EXACTLY this format:

OVERALL SENTIMENT: <Bullish/Bearish/Neutral> (<confidence>%)

KEY THEMES:
1. <theme> — <sentiment> — <explanation>
2. …
3. …
4. …
5. …

RISK SIGNALS:
- <risk>
- …

OPPORTUNITIES:
- <opportunity>
- …

TOP POSTS:
1. <platform> @<author>: <quote> — <why it matters>
2. …
3. …"""

In [ ]:
import time

client = anthropic.Anthropic()

# Retry on overload (529) errors
for attempt in range(5):
    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            messages=[{"role": "user", "content": prompt}],
        )
        break
    except anthropic.APIStatusError as e:
        if e.status_code in (429, 529) and attempt < 4:
            wait = 2 ** attempt + 1
            print(f"API overloaded, retrying in {wait}s...")
            time.sleep(wait)
        else:
            raise

analysis = response.content[0].text
print(analysis)

## Results

The analysis above shows Claude's structured interpretation of the social data. Let's also look at the highest-engagement posts that fed into the analysis.

In [ ]:
print("Top tweets by engagement:\n")
for t in sorted(tweets, key=lambda x: _safe_int(x.get("likeCount")), reverse=True)[:10]:
    likes = _safe_int(t.get("likeCount"))
    rts = _safe_int(t.get("retweetCount"))
    text = (t.get("text") or "")[:120]
    print(f"  @{t.get('authorUsername', '?')} ({likes:,} likes, {rts:,} RTs)")
    print(f"    {text}")
    print()

In [ ]:
print("Top Reddit posts by score:\n")
for p in sorted(reddit_posts, key=lambda x: _safe_int(x.get("score")), reverse=True)[:5]:
    score = _safe_int(p.get("score"))
    print(f"  r/{p.get('subredditName', '?')} — {p.get('title', '')} (score: {score:,})")
    print()

## How It Works

This cookbook demonstrates a pattern for building social intelligence pipelines:

1. **Data collection via MCP** — XPOZ provides a single protocol (MCP) to access Twitter, Reddit, Instagram and TikTok data. The JSON-RPC transport means you can call it from any HTTP client.

2. **Cross-platform aggregation** — By pulling from multiple platforms in parallel, you get a more complete picture than any single source provides.

3. **Structured analysis with Claude** — The prompt asks for a specific output format (sentiment, themes, risks, opportunities), making results consistent and actionable.

**Extending this pattern:**
- Add TikTok data via `getTiktokPostsByKeywords`
- Track sentiment over time by running this daily
- Compare two brands using parallel queries (e.g., Bitcoin vs Ethereum)
- Use Claude's tool_use to let it call XPOZ directly for follow-up queries

See the [XPOZ documentation](https://xpoz.ai) for the full list of available MCP tools.